# Day 1 — Document Ingestion
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 1 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

This notebook walks through the full Day 1 pipeline step by step: parsing a real clinical
guideline PDF, choosing a chunking strategy, generating embeddings, and building a
queryable vector index — the same steps implemented in `ingest.py`, but broken apart here
so you can inspect what happens at each stage before you rely on the script.

**By the end of this notebook you will be able to:**
1. Explain why grounding — not raw model memory — matters in clinical AI
2. Parse a PDF and inspect its extracted structure
3. Compare fixed-size vs. section-aware chunking on the same document
4. Generate an embedding and explain what the resulting vector represents
5. Build a persisted vector index and run a real query against it

> **Data source:** this notebook uses `data/WHO_Hypertension_Guideline_2021.pdf`, the real
> WHO guideline bundled with your starter kit — not a toy example.


## 0. Setup

Run this cell first. It adds the repo root to the path so we can reuse the exact same
functions defined in `ingest.py`, and confirms your environment is ready.


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config1
from pathlib import Path

print("Data directory:", config1.DATA_DIR)
print("Chunk size (tokens):", config1.CHUNK_SIZE)
print("Chunk overlap (tokens):", config1.CHUNK_OVERLAP)
print("PDFs found:", [p.name for p in config1.DATA_DIR.glob("*.pdf")])


Data directory: data
Chunk size (tokens): 100
Chunk overlap (tokens): 20
PDFs found: ['malaria_book_pages_1_114.pdf']


## 1. Why Grounding Matters

Before touching any code, sit with this for a second: a large language model can generate
a fluent, confident-sounding clinical recommendation **even when it has no real evidence
behind it.** It has no built-in mechanism to say "I don't know."

Retrieval-Augmented Generation (RAG) fixes this by separating two things:

- **What the model knows** (its training data — broad, but unverifiable and possibly stale)
- **What the model is allowed to say** (only what's in the text you hand it right now)

Everything you build today is the **first half** of that separation: turning a trustworthy
PDF into a searchable, citable index. Day 3 builds the second half (forcing the model to
answer only from what this index returns).


## 2. Step 1 — Parse the PDF

`PyPDFLoader` reads a PDF and returns one LangChain `Document` per page, each carrying
page-level metadata automatically (page number, source path).

Run the cell below and inspect the output. Notice that `page.metadata["page"]` is
**zero-indexed** — page index `0` is the PDF's first page.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
import config1
!pip install cryptography
pdf_path = list(config1.DATA_DIR.glob("*.pdf"))[0]
print(f"Loading: {pdf_path.name}\n")

loader = PyPDFLoader(str(pdf_path))
raw_pages = loader.load()

print(f"Loaded {len(raw_pages)} pages.\n")
print("--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---")
print(raw_pages[1].metadata)
print("\n--- Page 3 (index 2) first 400 characters ---")
print(raw_pages[1].page_content[:400])


/var/folders/4v/n8hg6dv94tbf2jw8tpbrlxth0000gn/T/ipykernel_45277/3446894039.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/habibaadawi/miniconda3/envs/medical_RAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading: malaria_book_pages_1_114.pdf

Loaded 114 pages.

--- Page 3 (index 2) raw metadata, as PyPDFLoader gives it to us ---
{'producer': 'Adobe PDF Library 11.0', 'creator': 'Adobe InDesign CC 2014 (Macintosh)', 'creationdate': '2015-06-08T17:07:13+02:00', 'moddate': '2015-06-09T16:10:13+02:00', 'trapped': '/False', 'source': 'data/malaria_book_pages_1_114.pdf', 'total_pages': 114, 'page': 1, 'page_label': 'ii'}

--- Page 3 (index 2) first 400 characters ---
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES


### Raw metadata isn't citation-ready yet

Look at the metadata above: PyPDFLoader gives you a generic `page` index and a full file
`source` path — but nothing called `document_name`, and no human-friendly 1-indexed page
number. If we chunk these pages as-is, every citation later would say "unknown, page ?".

`load_pdfs()` in `ingest.py` does one small but critical thing: it stamps
`document_name` and a 1-indexed `page_number` onto every page's metadata **before**
chunking, so that metadata survives all the way through to the final citation.


In [3]:
from ingest import load_pdfs

pages = load_pdfs(config1.DATA_DIR)

print("--- Page 3 (index 2) metadata AFTER normalization ---")
print({k: pages[1].metadata[k] for k in ["document_name", "page_number", "page"]})


Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES
WHO Library Cataloguing-in-Publication Data
Guidelines for the treatment of malaria – 3rd edition.
1.Malaria – drug therapy. 2.Malaria – diagnosis. 3.Antimalarials – administration and dosage. 4. 
Drug Therapy, Combination. 5.Guideline. I.World Health Organization.
ISBN 978 92 4 154912 7    (NLM classification: WC 770)
© World Health Organization 2015
All rights reserved. Publications of the World Health Organization are available on the WHO 
website (www.who.int) or can be purchased from WHO Press, World Health Organization, 
20 Avenue Appia, 1211 Geneva 27, Switzerland (tel.: +41 22 791 3264; fax: +41 22 791 4857; 
e-mail: bookorders@who.int). 
Requests for permission to reproduce or translate WHO publications –whether for sale or for non-
commercial distribution– should be addressed to WHO Press through the WHO website (www.
who.int/about/licensing/copyright_form/en/index.html

### Checkpoint 1

Look at the printed text above. Answer for yourself before moving on:

- Are section headings ("3.1 Blood pressure threshold...") visible as recognizable text, or
  did they get mangled?
- Are there any obvious parsing artifacts (broken words, merged columns, stray characters)?

If parsing looks clean here, section-aware chunking (Step 2) will work well. If it looks
messy, no chunking strategy will fully save you — the fix belongs upstream, in parsing.


## 3. Step 2 — Compare Chunking Strategies

We'll build **two** chunkers on the same pages and compare them directly: a naive
fixed-size splitter, and the section-aware splitter actually used in `ingest.py`.


In [4]:
#!-------------------------------!
# NOTE: You can use whatever Text splitter you prefer, e.g. (NLTK, spaCy)
#!-------------------------------!

from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Naive fixed-size splitter: no regard for sentence/paragraph boundaries ---
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # characters, not tokens — deliberately crude
    chunk_overlap=0,
    separators=[""],       # forces raw character-count splitting
)
naive_chunks = naive_splitter.split_documents(pages)

# --- Section-aware splitter: same one used in ingest.py ---
aware_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config1.CHUNK_SIZE * 4,      # ~4 chars/token estimate
    chunk_overlap=config1.CHUNK_OVERLAP * 4,
    separators=["\n\n", "\n", ". ", " ", ""],
)
aware_chunks = aware_splitter.split_documents(pages)

print(f"Naive fixed-size chunker:   {len(naive_chunks)} chunks")
print(f"Section-aware chunker:     {len(aware_chunks)} chunks")


Naive fixed-size chunker:   1155 chunks
Section-aware chunker:     720 chunks


In [5]:
# Look at one naive chunk boundary — notice it can cut mid-sentence
print("--- Naive chunk #5 (often cuts mid-sentence) ---")
print(repr(naive_chunks[1].page_content))

print("\n--- Section-aware chunk #5 (respects paragraph breaks) ---")
print(repr(aware_chunks[1].page_content[:300]))


--- Naive chunk #5 (often cuts mid-sentence) ---
'Third edition\nFOR THE TREATMENT\nOF MALARIA\nGUIDELINES'

--- Section-aware chunk #5 (respects paragraph breaks) ---
'Third edition\nFOR THE TREATMENT\nOF MALARIA\nGUIDELINES'


### Checkpoint 2

Compare the two printed chunks above.

- Does the naive chunk end mid-word or mid-sentence?
- Does the section-aware chunk end at a more natural paragraph or sentence boundary?

This is the entire argument for section-aware chunking in one comparison: **the boundary
you cut at becomes the boundary a citation has to point to.** A citation that lands mid-sentence
is much harder for a clinician to trust and verify.


## 4. Step 3 — Attach Citation Metadata

A chunk without a traceable source is useless for a clinical tool. Before embedding
anything, every chunk needs: **document name, page number, and a stable chunk id.**
This is exactly what `chunk_documents()` in `ingest.py` does — let's call it directly.


In [9]:
from ingest import chunk_documents

chunks = chunk_documents(pages)
print(f"Total chunks with metadata attached: {len(chunks)}\n")

sample = chunks[1]
print("--- Sample chunk metadata ---")
for k in ["document_name", "page_number", "chunk_id"]:
    print(f"  {k}: {sample.metadata.get(k)}")
print("\n--- Sample chunk text ---")
print(sample.page_content[:300])


Total chunks with metadata attached: 720

--- Sample chunk metadata ---
  document_name: malaria_book_pages_1_114.pdf
  page_number: 1
  chunk_id: None

--- Sample chunk text ---
Third edition
FOR THE TREATMENT
OF MALARIA
GUIDELINES


## 5. Step 4 — What Is an Embedding, Really?

An embedding model converts text into a list of numbers (a **vector**) that captures
meaning — texts about similar topics end up as vectors that point in similar directions,
even if they don't share any of the same words.

Let's embed three short phrases and check which two are "closer" in vector space.


In [6]:
import numpy as np
from ingest import get_embedding_function

embed_fn = get_embedding_function()

texts = [
    "first-line treatment for hypertension",
    "initial therapy for high blood pressure",   # means the same thing, different words
    "recommended screening interval for breast cancer",  # unrelated topic
]

vectors = embed_fn.embed_documents(texts)
vectors = np.array(vectors)
print(f"Each embedding is a vector of length {vectors.shape[1]}\n")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_related = cosine_similarity(vectors[0], vectors[1])
sim_unrelated = cosine_similarity(vectors[0], vectors[2])

print(f"Similarity — same meaning, different words:  {sim_related:.3f}")
print(f"Similarity — genuinely different topics:      {sim_unrelated:.3f}")


Each embedding is a vector of length 384

Similarity — same meaning, different words:  0.855
Similarity — genuinely different topics:      0.578


### Checkpoint 3

You should see the first similarity score noticeably **higher** than the second — the
"same meaning, different words" pair should score closer together, even though they share
almost no words in common. This is the entire mechanism semantic search relies on. If both
scores came out similar, something about the embedding model would be worth investigating
before trusting it on Day 2.


## 6. Step 5 — Build the Vector Index

Now we embed every chunk and store it in a local ChromaDB collection, using the exact
same `build_index()` function from `ingest.py`. This is the same call the script makes —
seeing it run here just makes the process visible.

> First run downloads a small local embedding model (~100MB) — this happens once and is
> cached afterward.


In [10]:
from ingest import build_index

vectordb = build_index(chunks)
print("\nIndex build complete.")



Index build complete.


In [12]:
print(chunks[200])

page_content='similar exposure across all patient groups. 
Weight-based dosage recommendations are summarized below. While age-based 
dosing may be more practical in children, the relation between age and weight 
differs in different populations. Age-based dosing can therefore result in under-
dosing or over-dosing of some patients, unless large, region-specific weight-for-age' metadata={'document_name': 'malaria_book_pages_1_114.pdf', 'page_number': 37, 'page': '4.3.3 | DOSING OF ACTS\nACT regimens must ensure optimal dosing to prolong their useful therapeutic life, \ni.e. to maximize the likelihood of rapid clinical and parasitological cure, minimize \ntransmission and retard drug resistance.\nIt is essential to achieve effective antimalarial drug concentrations for a sufficient \ntime (exposure) in all target populations in order to ensure high cure rates. \nThe dosage recommendations below are derived from understanding the relationship \nbetween dose and the profiles of exposure t

## 7. Step 6 — Run a Real Query

The whole point of everything above: ask a real clinical question and see whether the
index returns something relevant.


In [16]:
malaria_treatment_qa = [
    {
        "question": "What are the recommended artemisinin-based combination therapies (ACTs) for treating uncomplicated P. falciparum malaria in children and adults?",
        "answer": "The five recommended ACTs are: artemether + lumefantrine, artesunate + amodiaquine, artesunate + mefloquine, dihydroartemisinin + piperaquine, and artesunate + sulfadoxine-pyrimethamine (SP). These are recommended for children and adults except pregnant women in their first trimester.",
        "page": 35,
        "section": "4.3.1 Artemisinin-based combination therapy"
    },
    {
        "question": "What is the recommended duration of ACT treatment for uncomplicated P. falciparum malaria?",
        "answer": "ACT regimens should provide 3 days' treatment with an artemisinin derivative. A 3-day course covers two asexual cycles, ensuring only a small fraction of parasites remain for clearance by the partner drug, thus reducing the potential development of resistance.",
        "page": 37,
        "section": "4.3.2 Duration of treatment"
    },
    {
        "question": "How should uncomplicated P. falciparum malaria be treated in pregnant women during the first trimester?",
        "answer": "Pregnant women with uncomplicated P. falciparum malaria during the first trimester should be treated with 7 days of quinine + clindamycin (10mg/kg bw twice a day). An ACT or oral artesunate + clindamycin is an alternative if quinine + clindamycin is not available or fails.",
        "page": 52,
        "section": "5.1.1 First trimester"
    },
    {
        "question": "What is the recommended treatment for severe malaria in adults and children?",
        "answer": "Treat adults and children with severe malaria (including infants, pregnant women in all trimesters and lactating women) with intravenous or intramuscular artesunate for at least 24 hours and until they can tolerate oral medication. Once a patient has received at least 24 hours of parenteral therapy and can tolerate oral therapy, complete treatment with 3 days of an ACT.",
        "page": 79,
        "section": "7.4.1 Artesunate"
    },
    {
        "question": "What is the revised dose recommendation for parenteral artesunate in young children with severe malaria?",
        "answer": "Children weighing less than 20kg should receive a higher parenteral dose of artesunate (3 mg/kg per dose) than larger children and adults (2.4 mg/kg per dose) to ensure equivalent drug exposure. This is because young children have a larger apparent volume of distribution for both compounds.",
        "page": 80,
        "section": "7.4.1 Artesunate"
    },
    {
        "question": "What is the recommended pre-referral treatment for suspected severe malaria when complete treatment is not possible?",
        "answer": "Where complete treatment of severe malaria is not possible but injections are available, give adults and children a single intramuscular dose of artesunate, and refer to an appropriate facility for further care. Where intramuscular artesunate is not available, use intramuscular artemether or, if that is not available, use intramuscular quinine.",
        "page": 84,
        "section": "7.5 Pre-referral treatment options"
    },
    {
        "question": "How should uncomplicated P. vivax malaria be treated in areas with chloroquine-resistant infections?",
        "answer": "In areas with chloroquine-resistant infections, treat adults and children with uncomplicated P. vivax, P. ovale, P. malariae or P. knowlesi malaria (except pregnant women in their first trimester) with an ACT. ACTs containing piperaquine, mefloquine or lumefantrine are the recommended treatment.",
        "page": 67,
        "section": "6.4.1 Uncomplicated P. vivax malaria"
    },
    {
        "question": "What is the recommended regimen for preventing relapse in P. vivax or P. ovale malaria?",
        "answer": "To prevent relapse, treat P. vivax or P. ovale malaria in children and adults (except pregnant women, infants aged < 6 months, women breastfeeding infants < 6 months, and people with G6PD deficiency) with a 14-day course (0.25-0.5 mg/kg bw daily) of primaquine in all transmission settings. Total doses of 3.5 mg base/kg bw are required for temperate strains and 7 mg base/kg bw for tropical strains.",
        "page": 69,
        "section": "6.5.1 Primaquine for preventing relapse"
    },
    {
        "question": "What is the recommended single dose of primaquine for reducing transmission of P. falciparum in low-transmission areas?",
        "answer": "In low-transmission areas, give a single dose of 0.25mg/kg bw primaquine with ACT to patients with P. falciparum malaria (except pregnant women, infants aged < 6 months and women breastfeeding infants aged < 6 months) to reduce transmission. G6PD testing is not required for this indication as this dose is unlikely to cause serious toxicity.",
        "page": 45,
        "section": "4.5 Reducing the transmissibility of treated P. falciparum infections"
    },
    {
        "question": "What parenteral alternatives are recommended when artesunate is not available for treating severe malaria?",
        "answer": "If parenteral artesunate is not available, use intramuscular artemether in preference to quinine for treating children and adults with severe malaria. Artemether is given as an initial dose of 3.2mg/kg bw intramuscularly, with a maintenance dose of 1.6mg/kg bw daily. If artemether is also unavailable, use parenteral quinine.",
        "page": 81,
        "section": "7.4.2 Parenteral alternatives when artesunate is not available"
    }
]


In [ ]:
import numpy as np
import config1
import chromadb



def evaluate_retrieval(qa_dataset, collection, embedder, k=3, page_offset=0):
    """
    Evaluates Precision@K and Recall@K for your Chroma collection using 
    fastembed model embeddings and metadata fields.
    
    Parameters:
    - page_offset: Set to 1 if `PyPDFLoader` index `page_number` starts at 0 
                   while QA target page numbers start at 1.
    """
    precisions = []
    recalls = []

    print(f"=== Running Retrieval Evaluation (k={k}) ===\n")

    for idx, sample in enumerate(qa_dataset, 1):
        question = sample["question"]
        expected_page = sample["page"]

        # 1. Generate query embedding using your custom wrapper
        query_vector = embedder.embed_documents([question])[0].tolist()

        # 2. Query Chroma collection
        search_results = collection.query(
            query_embeddings=[query_vector],
            n_results=k,
            include=["metadatas"]
        )

        retrieved_metadatas = search_results["metadatas"][0]

        # 3. Extract page_number matching your Page metadata definition
        retrieved_pages = [
            meta.get("page_number") + page_offset if meta.get("page_number") is not None else None
            for meta in retrieved_metadatas
        ]

        # 4. Count relevant page hits
        hit_count = sum(1 for page in retrieved_pages if page == expected_page)

        # Precision@K: Ratio of retrieved chunks matching the target page
        precision_at_k = hit_count / k

        # Recall@K: 1.0 if target page is retrieved within top-k, else 0.0
        recall_at_k = 1.0 if hit_count > 0 else 0.0

        precisions.append(precision_at_k)
        recalls.append(recall_at_k)

        print(f"[{idx}/{len(qa_dataset)}] Question: {question[:65]}...")
        print(f"    Target Page: {expected_page} | Retrieved Pages: {retrieved_pages}")
        print(f"    Precision@{k}: {precision_at_k:.2f} | Recall@{k}: {recall_at_k:.2f}\n")

    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)

    print("=" * 45)
    print(f"Mean Precision@{k}: {mean_precision:.4f} ({mean_precision * 100:.2f}%)")
    print(f"Mean Recall@{k}:    {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("=" * 45)

    return {"mean_precision": mean_precision, "mean_recall": mean_recall}


# --- Execution Example ---
if __name__ == "__main__":
    # Fetch collection configuration from config1
    collection_name = getattr(config1, "COLLECTION_NAME", "documents")
    chroma_client = chromadb.PersistentClient(path="./chroma_db")
    collection = chroma_client.get_collection(name=collection_name)

    # Instantiate your custom embedding wrapper from the file above
    embedder = get_embedding_function()

    # Evaluate Precision@3 and Recall@3
    # Note: Pass page_offset=1 if PyPDFLoader starts page_number at 0 instead of 1
    metrics = evaluate_retrieval(
        qa_dataset=malaria_treatment_qa,
        collection=collection,
        embedder=embedder,
        k=4,
        page_offset=0
    )

=== Running Retrieval Evaluation (k=4) ===

[1/10] Question: What are the recommended artemisinin-based combination therapies ...
    Target Page: 35 | Retrieved Pages: [11, 18, 35, 34]
    Precision@4: 0.25 | Recall@4: 1.00

[2/10] Question: What is the recommended duration of ACT treatment for uncomplicat...
    Target Page: 37 | Retrieved Pages: [10, 60, 59, 34]
    Precision@4: 0.00 | Recall@4: 0.00

[3/10] Question: How should uncomplicated P. falciparum malaria be treated in preg...
    Target Page: 52 | Retrieved Pages: [12, 50, 51, 52]
    Precision@4: 0.25 | Recall@4: 1.00

[4/10] Question: What is the recommended treatment for severe malaria in adults an...
    Target Page: 79 | Retrieved Pages: [73, 74, 74, 14]
    Precision@4: 0.00 | Recall@4: 0.00

[5/10] Question: What is the revised dose recommendation for parenteral artesunate...
    Target Page: 80 | Retrieved Pages: [79, 13, 79, 78]
    Precision@4: 0.00 | Recall@4: 0.00

[6/10] Question: What is the recommended pre-r

### Checkpoint 4 — Day 1 Self-Check

Before you close this notebook, confirm all of the following are true:

- [true ] The top retrieved chunk in Step 6 is genuinely relevant to the question asked
- [true ] Every result shows a document name and page number (not `None`)
- [ true] You could explain to a teammate, in one sentence, why section-aware chunking beat
      the naive splitter in Checkpoint 2

If any of these aren't true yet, that's normal — go back to the relevant step above and
adjust `config.py` (chunk size, overlap) before moving on to Day 2.

## What's Next

Day 2's notebook picks up exactly here: tuning `top_k`, benchmarking this embedding model
against alternatives, and proving your retrieval quality with real, logged numbers instead
of a single example query.


### Testing the LLM 

In [20]:
!pip install langchain_chroma
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from ingest import get_embedding_function

# 1. Initialize the PubMedBERT embedding function from ingest.py
embedding_fn = get_embedding_function()

# 2. Load the existing local vector database index
vector_db = Chroma(
    persist_directory="./chroma_db",  # Update path if your ChromaDB folder is named differently
    embedding_function=embedding_fn
)

# 3. Initialize your local LLM (e.g., via Ollama)
local_llm = Ollama(model="medllama2:latest")  # Change model name to match your local setup

# 4. Create retriever interface
retriever = vector_db.as_retriever(search_kwargs={"k": 3})


print("--- Running Local RAG Evaluation ---")
for item in eval_dataset:
    question = item["question"]
    print(f"\n[Q{item['id']}] {question}")
    
    # Retrieve relevant context chunks using vector index
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Construct RAG prompt
    prompt = f"""Use the provided context to answer the medical question accurately.

Context:
{context}

Question: {question}
Answer:"""
    
    # Generate response from local LLM
    generated_answer = local_llm.invoke(prompt)
    
    print(f"-> Generated: {generated_answer.strip()}")
    print(f"-> Expected:  {item['expected_answer']}")
    print("=" * 60)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10487.47it/s]


--- Running Local RAG Evaluation ---

[Q1] According to the chapter on "Biliary Obstruction," what is the single most useful noninvasive procedure for differentiating intrahepatic cholestasis from extrahepatic obstruction?
-> Generated: This question is nonsensical because the two conditions are not distinguishable based on a single noninvasive procedure. The differential diagnosis of jaundice is complex and requires a multidisciplinary approach. Further evaluation is needed to determine the underlying cause.
-> Expected:  Ultrasonography

[Q2] Describe the mechanism by which HbA1c (glycosylated hemoglobin) provides a measure of long-term diabetic control. How is it formed, and what does its level reflect?
-> Generated: Hb A, is a measure of the average blood glucose levels over the previous two to three months. It reflects the integrated level of blood glucose over this period. Its measurement provides a useful index of cumulative control of hyperglycemia during this time, and can be 

KeyboardInterrupt: 

### Testing citation accuracy 